# 00 · DIV2K operator-mismatch pilot
**Experiment log · 17 September 2026 · Four development source images**

Working direction: *Operator-Sensitive Selective Reconstruction Under Forward-Model Mismatch*.
This is the initial data/physics/metric check for the broader physics-informed reconstruction study.

## tl;dr
All four source images passed decoding, opaque-alpha and duplicate checks. The reproducible CPU pipeline ran through two degradation cases.

For blur + noise, pooled-MSE PSNR was **29.74 dB** for the degraded input,
**25.19 dB** for the wrong-blur inverse and **29.10 dB** for the known-blur inverse.
**Both fixed inverses lost to the degraded input.** The baseline needs development before deeper model comparisons.

At 50% pixel retention, operator-sensitivity ranking reduced MSE versus expected random selection by
**9.9%** (blur + noise) and **40.2%** (JPEG stress).
It did **not** beat the best simple control at that coverage in either case.
These four-image observations establish a working test harness, not research validation or novelty.



## Context & Methods
We test whether small changes in an assumed blur operator reveal unreliable reconstruction regions.
The simulation is $y=A_{1.6}x+\epsilon$, where $x$ is the stored clean RGB reference and
$\epsilon\sim\mathcal{N}(0,(2/255)^2)$. The nominal inverse assumes blur width **1.0 pixels**.
A second inverse receives the true blur width **1.6 pixels** as a diagnostic, not an estimated parameter.

The classical inverse solves $\min_x\|A_\sigma x-y\|_2^2+\lambda\|x\|_2^2$ with fixed $\lambda=0.002$.
With Fourier response $H_\sigma$, the solution is
$\hat{x}=\mathcal{F}^{-1}\{\overline{H_\sigma}\mathcal{F}(y)/(|H_\sigma|^2+\lambda)\}$.
Reconstructions are then clipped to [0,1] for scoring; this clipping is a separate postprocessing step.

### Key assumptions
- New pilot settings are engineering defaults selected for this check, not tuned or claimed optimal.
- One 576 × 576 native-resolution centre crop per image supplies a 512 × 512 evaluation region with a 32-pixel context margin.
- The Fourier blur has periodic boundaries. The margin reduces wraparound effects; this is not a validated camera model.
- Computation is in stored gamma-encoded RGB, not calibrated linear-light radiometry.
- The linear observation stays floating point and is not clipped before inversion. Only its display/baseline is clipped.
- A separate stress case adds clipping, 8-bit quantisation and JPEG quality 75 (no chroma subsampling).
  Even the known-blur inverse is incomplete for this nonlinear JPEG case.
- Ground-truth references are used for evaluation only, except the explicitly labelled oracle diagnostic.
- This pilot has no neural model, calibrated uncertainty, independent test set, or claim of novel performance.

### Scores and endpoints
Operator sensitivity is the RMS across colour channels of reconstruction standard deviations under
blur widths {0.8, 1.0, 1.2} times the nominal width. It measures **perturbation spread**, not a probability.
Controls use measurement residual and image-gradient magnitude (an image-only heuristic, not an uncertainty estimate).
Risk–coverage curves retain the lowest-scoring pixels within each image; risk is mean RGB squared error.
Random selection has expected risk equal to the full-image mean. An oracle sorts by actual reference error.
Curves are averaged equally over four source images. Pixels and repeated degradations are not independent source samples.
PSNR uses range 1; pooled-MSE PSNR and mean image PSNR are reported separately.


## Data
User-supplied PNGs: **0801–0804**. Their names match the DIV2K validation numbering;
their upstream canonical checksums have not been independently verified.
The [official DIV2K page](https://data.vision.ee.ethz.ch/cvl/DIV2K/) documents 800 training images
and 100 validation HR images, numbered 0801–0900. That validation folder alone cannot provide the planned
100 calibration sources plus 100 separate test sources and development data.

These four images, all derivatives and crops belong to **development only** from now on.
No source is assigned to calibration or final testing in this notebook.
Preserve image-level disjointness and audit pretrained-model training data before choosing final evaluation sources.
Source filenames, byte hashes, pixel hashes, dimensions and crop coordinates are saved in `results/manifest.csv`.

### 1. Setup
Open this notebook from the extracted starter folder, which contains `samples/`, or point `DATA_DIR`
below to your own extracted `DIV2K_valid_HR` directory. All code is contained in this notebook.
In Colab, upload the four PNGs into `/content/samples/` and set `DATA_DIR` accordingly.
Only CPU execution is required.

If a dependency is missing, install `numpy pandas scipy matplotlib Pillow` in your Python environment.
For a local notebook interface, install JupyterLab separately. Sources and output directories must be separate.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
"/content/drive/MyDrive/reliable-reconstruction-under-mismatch/samples"

'/content/drive/MyDrive/reliable-reconstruction-under-mismatch/samples'

### 2. Audit source files
Reject corrupt files and nonopaque alpha, decode RGB, check dimensions and preserve the original bytes.
The uploaded four images have alpha values 255 throughout, so removing that channel preserves their colour values.


In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_source(path, config):
    """Audit bytes, reject nonopaque alpha, and take one native-resolution crop."""
    path = Path(path)
    with Image.open(path) as image:
        image.verify()
    with Image.open(path) as image:
        image.load()
        mode, width, height = image.mode, image.width, image.height
        if image.format != "PNG" or mode not in ("RGB", "RGBA"):
            raise ValueError(f"{path.name}: expected an 8-bit RGB/RGBA PNG, got {mode}")
        alpha_min = alpha_max = None
        if mode == "RGBA":
            alpha_min, alpha_max = image.getchannel("A").getextrema()
            if (alpha_min, alpha_max) != (255, 255):
                raise ValueError(f"{path.name}: nonopaque alpha needs an explicit policy")
        rgb = np.asarray(image.convert("RGB"), dtype=np.float64) / 255.0
    extent = config["crop_size"] + 2 * config["context_border"]
    if min(height, width) < extent:
        raise ValueError(f"{path.name}: too small for a {extent} x {extent} context crop")
    left, top = (width - extent) // 2, (height - extent) // 2
    patch = rgb[top:top + extent, left:left + extent].copy()
    record = {
        "source_id": path.stem, "filename": path.name,
        "width": width, "height": height, "original_mode": mode,
        "alpha_min": alpha_min, "alpha_max": alpha_max,
        "size_bytes": path.stat().st_size, "sha256": sha256_file(path),
        "rgb_sha256": hashlib.sha256((rgb * 255).round().astype("uint8").tobytes()).hexdigest(),
        "role": "development_only", "crop_left": left, "crop_top": top,
        "context_extent": extent, "evaluated_extent": config["crop_size"],
        "status": "decoded",
    }
    return patch, record


In [5]:
import argparse
import hashlib
import io
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
from PIL import Image
import scipy
from scipy.stats import spearmanr


import os

CONFIG = {
    "source_ids": ["0801", "0802", "0803", "0804"],
    "role": "development_only",
    "seed": 20260917,
    "crop_size": 512,
    "context_border": 32,
    "true_blur_sigma_px": 1.6,
    "assumed_blur_sigma_px": 1.0,
    "noise_std": 2.0 / 255.0,
    "ridge_lambda": 0.002,
    "perturbation_fraction": 0.2,
    "jpeg_quality": 75,
    "jpeg_subsampling": 0,
    "coverage": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "scenarios": ["blur_noise", "blur_noise_jpeg"],
    "operator": "periodic Fourier Gaussian; evaluation excludes context border",
    "metric": "RGB MSE on [0,1]; PSNR uses data range 1",
    "colour_space": "stored gamma-encoded RGB; not a linear-light sensor model",
}


# Change this path only if your source folder is somewhere else.
DATA_DIR = Path("/content/drive/MyDrive/reliable-reconstruction-under-mismatch/samples")
OUTPUT_DIR = Path("results")
assert DATA_DIR.is_dir(), f"Set DATA_DIR to the folder containing 0801.png–0804.png: {DATA_DIR.resolve()}"
assert DATA_DIR.resolve() != OUTPUT_DIR.resolve(), "Keep sources and results in separate folders."
print("Development sources:", ", ".join(CONFIG["source_ids"]))
print("Source folder:", DATA_DIR.resolve())


Development sources: 0801, 0802, 0803, 0804
Source folder: /content/drive/MyDrive/reliable-reconstruction-under-mismatch/samples


### 3. Define degradation and reconstruction
The same noise realisation feeds both the linear and JPEG cases for each source. Seeds depend on source identity,
so changing the order of the sources does not change the simulated observations.


In [6]:
def gaussian_transfer(shape, sigma):
    """Analytic Gaussian frequency response; sigma is in working-image pixels."""
    fy = np.fft.fftfreq(shape[0])[:, None]
    fx = np.fft.fftfreq(shape[1])[None, :]
    return np.exp(-2.0 * np.pi**2 * sigma**2 * (fx**2 + fy**2))


def apply_operator(image, sigma):
    transfer = gaussian_transfer(image.shape[:2], sigma)
    spectrum = np.fft.fft2(image, axes=(0, 1))
    return np.fft.ifft2(spectrum * transfer[..., None], axes=(0, 1)).real


def reconstruct(measurement, sigma, ridge_lambda, clip=True):
    """Solve min_x ||A_sigma x-y||_2^2 + lambda ||x||_2^2 by FFT."""
    transfer = gaussian_transfer(measurement.shape[:2], sigma)
    inverse = transfer / (transfer**2 + ridge_lambda)
    estimate = np.fft.ifft2(
        np.fft.fft2(measurement, axes=(0, 1)) * inverse[..., None], axes=(0, 1)
    ).real
    return np.clip(estimate, 0, 1) if clip else estimate


def simulate_measurements(reference, source_id, config):
    """Use identical sensor noise for the linear and JPEG variants of one source."""
    identity = int(hashlib.sha256(source_id.encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(np.random.SeedSequence([config["seed"], identity]))
    linear = apply_operator(reference, config["true_blur_sigma_px"])
    linear += rng.normal(0, config["noise_std"], size=reference.shape)
    # Keep the linear observation unbounded. JPEG explicitly adds clipping,
    # 8-bit quantisation, and nonlinear compression: a separate stress case.
    quantised = np.rint(np.clip(linear, 0, 1) * 255).astype("uint8")
    buffer = io.BytesIO()
    Image.fromarray(quantised).save(
        buffer, format="JPEG", quality=config["jpeg_quality"],
        subsampling=config["jpeg_subsampling"], optimize=False
    )
    buffer.seek(0)
    with Image.open(buffer) as compressed:
        jpeg = np.asarray(compressed.convert("RGB"), dtype=np.float64) / 255.0
    return {"blur_noise": linear, "blur_noise_jpeg": jpeg}


def interior(array, config):
    border = config["context_border"]
    return array[border:-border, border:-border] if border else array


def psnr_from_mse(mse):
    return float(-10 * np.log10(mse)) if mse > 0 else float("inf")


### 4. Define sensitivity scores and selective error
The three operational score functions receive no clean image. The clean reference enters only the error calculation
and the explicitly labelled oracle. Rank correlation is descriptive; pixelwise significance tests are not used.


In [7]:
def score_maps(measurement, assumed_sigma, config):
    """All operational scores use only y, the assumed operator, and its inverse."""
    delta = config["perturbation_fraction"]
    sigmas = np.array([1 - delta, 1, 1 + delta]) * assumed_sigma
    estimates = np.stack([
        reconstruct(measurement, value, config["ridge_lambda"]) for value in sigmas
    ])
    nominal = estimates[1]
    spread = np.sqrt(np.mean(np.var(estimates, axis=0, ddof=0), axis=-1))
    residual = np.sqrt(np.mean(
        (apply_operator(nominal, assumed_sigma) - measurement)**2, axis=-1
    ))
    grey = nominal.mean(axis=-1)
    gy, gx = np.gradient(grey)
    gradient = np.hypot(gx, gy)
    return nominal, {
        "operator_spread": spread,
        "measurement_residual": residual,
        "image_gradient": gradient,
    }


def selective_curve(error_map, score, coverages, seed):
    """Retain lowest-scoring pixels; seeded tie breaks avoid raster-order bias."""
    error, score = error_map.ravel(), score.ravel()
    rng = np.random.default_rng(seed)
    tie_break = rng.random(score.size)
    order = np.lexsort((tie_break, score))
    cumulative = np.cumsum(error[order], dtype=np.float64)
    out = []
    for coverage in coverages:
        count = max(1, int(np.ceil(coverage * error.size)))
        risk = float(cumulative[count - 1] / count)
        out.append({"coverage": coverage, "retained_pixels": count,
                    "mse": risk, "psnr_db": psnr_from_mse(risk)})
    return out


### 5. Validate the mathematics
Check constant preservation, the forward/adjoint relation, the ridge normal equation,
an independently hand-checkable selection example and the PSNR scale.


In [8]:
def numerical_checks(config):
    """Check inverse mathematics and selection endpoints independently of images."""
    rng = np.random.default_rng(18)
    x, z = rng.random((32, 32, 3)), rng.random((32, 32, 3))
    sigma = config["true_blur_sigma_px"]
    constant = np.ones_like(x)
    assert np.allclose(apply_operator(constant, sigma), constant, atol=1e-12)
    lhs = np.vdot(apply_operator(x, sigma), z)
    rhs = np.vdot(x, apply_operator(z, sigma))
    assert np.allclose(lhs, rhs, atol=1e-10)
    estimate = reconstruct(x, sigma, config["ridge_lambda"], clip=False)
    normal_residual = apply_operator(apply_operator(estimate, sigma) - x, sigma)
    normal_residual += config["ridge_lambda"] * estimate
    assert np.max(np.abs(normal_residual)) < 1e-10
    error = np.array([[0.1, 0.2], [0.3, 0.4]])
    curve = selective_curve(error, error, [0.5, 1.0], 1)
    assert np.isclose(curve[0]["mse"], 0.15)
    assert np.isclose(curve[1]["mse"], error.mean())
    assert np.isclose(psnr_from_mse(0.01), 20.0)
    return {"constant_preserved": True, "adjoint_check": True,
            "ridge_normal_equation_max_abs": float(np.abs(normal_residual).max()),
            "selection_endpoint_check": True, "psnr_scale_check": True}


In [9]:
print(json.dumps(numerical_checks(CONFIG), indent=2))


{
  "constant_preserved": true,
  "adjoint_check": true,
  "ridge_normal_equation_max_abs": 2.8189256484623115e-16,
  "selection_endpoint_check": true,
  "psnr_scale_check": true
}


### 6. Run the four-source development pilot
This saves the configuration, manifest, metrics, score correlations, risk–coverage points and validation checks.
The original PNGs are never overwritten. Failed audits stop execution.


In [10]:
def run_pilot(data_dir, output_dir, config=None):
    config = json.loads(json.dumps(CONFIG if config is None else config))
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checks = numerical_checks(config)
    manifest, metrics, curves, correlations, examples = [], [], [], [], []
    source_hashes = set()
    for source_index, source_id in enumerate(config["source_ids"]):
        reference, record = load_source(data_dir / f"{source_id}.png", config)
        if record["rgb_sha256"] in source_hashes:
            raise ValueError(f"Duplicate RGB source content: {source_id}")
        source_hashes.add(record["rgb_sha256"])
        manifest.append(record)
        truth = interior(reference, config)
        observations = simulate_measurements(reference, source_id, config)
        for scenario in config["scenarios"]:
            measurement = observations[scenario]
            estimates = {"observed": np.clip(measurement, 0, 1)}
            scores_by_model = {}
            for model, assumed_sigma in [
                ("assumed_blur", config["assumed_blur_sigma_px"]),
                ("known_blur_diagnostic", config["true_blur_sigma_px"]),
            ]:
                estimates[model], scores_by_model[model] = score_maps(
                    measurement, assumed_sigma, config
                )
            for model, estimate in estimates.items():
                error_map = np.mean((interior(estimate, config) - truth)**2, axis=-1)
                full_mse = float(error_map.mean())
                metrics.append({"source_id": source_id, "scenario": scenario,
                                "model": model, "mse": full_mse,
                                "psnr_db": psnr_from_mse(full_mse)})
                if model == "observed":
                    continue
                for score_name, score in scores_by_model[model].items():
                    central_score = interior(score, config)
                    rho = spearmanr(central_score.ravel(), error_map.ravel()).statistic
                    correlations.append({"source_id": source_id, "scenario": scenario,
                                         "model": model, "score": score_name,
                                         "spearman_rho": float(rho)})
                    for point in selective_curve(error_map, central_score,
                                                 config["coverage"],
                                                 config["seed"] + source_index):
                        curves.append({"source_id": source_id, "scenario": scenario,
                                       "model": model, "score": score_name, **point})
                for point in selective_curve(error_map, error_map, config["coverage"],
                                             config["seed"] + source_index):
                    curves.append({"source_id": source_id, "scenario": scenario,
                                   "model": model, "score": "oracle_reference_error", **point})
                for coverage in config["coverage"]:
                    curves.append({"source_id": source_id, "scenario": scenario,
                                   "model": model, "score": "random_expected",
                                   "coverage": coverage,
                                   "retained_pixels": int(np.ceil(coverage * error_map.size)),
                                   "mse": full_mse, "psnr_db": psnr_from_mse(full_mse)})
            if scenario == "blur_noise":
                examples.append({
                    "source_id": source_id, "reference": truth,
                    "observation": interior(np.clip(measurement, 0, 1), config),
                    "assumed": interior(estimates["assumed_blur"], config),
                    "known": interior(estimates["known_blur_diagnostic"], config),
                    "error": np.sqrt(np.mean((interior(estimates["assumed_blur"], config)-truth)**2, axis=-1)),
                    "spread": interior(scores_by_model["assumed_blur"]["operator_spread"], config),
                })
                # Measurements remain float64 in memory; raw observations are never
                # reconstructed from display PNGs. Source bytes, deterministic seeds,
                # configuration and environment versions support regeneration.
    metrics, curves = pd.DataFrame(metrics), pd.DataFrame(curves)
    correlations, manifest = pd.DataFrame(correlations), pd.DataFrame(manifest)
    summary = metrics.groupby(["scenario", "model"], as_index=False).agg(
        mean_mse=("mse", "mean"), mean_image_psnr_db=("psnr_db", "mean"),
        source_images=("source_id", "nunique"))
    summary["psnr_from_pooled_mse_db"] = summary["mean_mse"].map(psnr_from_mse)
    for name, frame in [("manifest", manifest), ("metrics", metrics), ("risk_coverage", curves),
                        ("score_correlations", correlations), ("summary", summary)]:
        frame.to_csv(output_dir / f"{name}.csv", index=False)
    for record in manifest.to_dict("records"):
        assert sha256_file(data_dir / record["filename"]) == record["sha256"]
    endpoint = curves[curves.coverage == 1].merge(
        metrics[["source_id", "scenario", "model", "mse"]],
        on=["source_id", "scenario", "model"], suffixes=("_selected", "_full"))
    assert np.allclose(endpoint.mse_selected, endpoint.mse_full)
    # An oracle sorting by actual error must not have greater selective risk.
    oracle = curves[curves.score == "oracle_reference_error"].set_index(
        ["source_id", "scenario", "model", "coverage"])["mse"]
    for _, row in curves.iterrows():
        key = (row.source_id, row.scenario, row.model, row.coverage)
        assert oracle.loc[key] <= row.mse + 1e-12
    checks.update({"source_files_unchanged": True, "no_duplicate_rgb_sources": True,
                   "all_curves_reconcile_at_full_coverage": True,
                   "oracle_lower_bound_check": True,
                   "source_images": len(manifest), "metrics_rows": len(metrics),
                   "scope": "four development images; no independent inference"})
    environment = {"python": platform.python_version(), "numpy": np.__version__,
                   "pandas": pd.__version__, "scipy": scipy.__version__,
                   "pillow": PIL.__version__, "matplotlib": matplotlib.__version__}
    for name, payload in [("config", config), ("checks", checks), ("environment", environment)]:
        (output_dir / f"{name}.json").write_text(json.dumps(payload, indent=2) + "\n")
    return {"config": config, "manifest": manifest, "metrics": metrics,
            "curves": curves, "correlations": correlations, "summary": summary,
            "examples": examples, "checks": checks, "environment": environment}


In [13]:
result = run_pilot(DATA_DIR, OUTPUT_DIR, CONFIG)
print(result["manifest"][["filename", "width", "height", "original_mode", "role"]].to_string(index=False))


filename  width  height original_mode             role
0801.png   2040    1356          RGBA development_only
0802.png   2040    1356          RGBA development_only
0803.png   2040    1536          RGBA development_only
0804.png   2040    1200          RGBA development_only


## Results
### 7. Compare reconstruction quality
Higher PSNR is better. The observed-image baseline must remain visible: an inverse that loses to it needs development work.
All numbers below are descriptive for these four centre crops.


In [14]:
def describe_results(result):
    summary = result["summary"]
    print("FOUR-IMAGE DEVELOPMENT PILOT — no independent test conclusions\n")
    print(summary.round(5).to_string(index=False))
    frame = result["curves"]
    at_half = frame[(frame.coverage == .5) & (frame.model == "assumed_blur")]
    rows = at_half.groupby(["scenario", "score"])["mse"].mean().unstack()
    for scenario, values in rows.iterrows():
        improvement = 100 * (1 - values.operator_spread / values.random_expected)
        print(f"\n{scenario}: at 50% retained pixels, operator-sensitivity ranking changes "
              f"mean squared error by {improvement:+.1f}% improvement versus random selection.")
        deployable = values[["operator_spread", "measurement_residual", "image_gradient"]]
        print(f"Lowest retained error among the three operational scores: {deployable.idxmin()}.")
    print("\nThese are descriptive development comparisons. Known blur and oracle error use simulation truth.")
    print("Pretrained deep models, calibrated thresholds, and independent source-image testing remain future stages.")


In [15]:
describe_results(result)


FOUR-IMAGE DEVELOPMENT PILOT — no independent test conclusions

       scenario                 model  mean_mse  mean_image_psnr_db  source_images  psnr_from_pooled_mse_db
     blur_noise          assumed_blur   0.00302            25.25972              4                 25.19469
     blur_noise known_blur_diagnostic   0.00123            29.21314              4                 29.10481
     blur_noise              observed   0.00106            30.46080              4                 29.74379
blur_noise_jpeg          assumed_blur   0.00170            27.99912              4                 27.69682
blur_noise_jpeg known_blur_diagnostic   0.00155            28.30083              4                 28.10548
blur_noise_jpeg              observed   0.00108            30.39887              4                 29.67759

blur_noise: at 50% retained pixels, operator-sensitivity ranking changes mean squared error by +9.9% improvement versus random selection.
Lowest retained error among the three ope

### 8. Inspect reconstructions, selective-error curves and sensitivity maps
Each curve averages the same four source images. The two scenario rows have separate y-axis ranges,
but the assumed/known-blur columns within each row share a scale. Error/sensitivity heatmaps use a common scale within each row.


In [16]:
COLOURS = {"operator_spread": "#176B9B", "measurement_residual": "#C76E23",
           "image_gradient": "#8B5C94", "random_expected": "#555555",
           "oracle_reference_error": "#9A852C"}


LABELS = {"operator_spread": "Operator sensitivity", "measurement_residual": "Measurement residual",
          "image_gradient": "Image-gradient heuristic", "random_expected": "Random selection (expected)",
          "oracle_reference_error": "Oracle (uses clean reference)"}


STYLES = {"operator_spread": "-", "measurement_residual": "--", "image_gradient": "-.",
          "random_expected": ":", "oracle_reference_error": "--"}


def plot_results(result, output_dir):
    output_dir = Path(output_dir)
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "figure.facecolor": "white", "savefig.facecolor": "white"})
    figures = []
    examples = result["examples"]
    fig, axes = plt.subplots(len(examples), 4, figsize=(12, 3.15 * len(examples)), squeeze=False)
    for row, example in enumerate(examples):
        for col, (key, title) in enumerate([
            ("reference", "Clean reference"), ("observation", "Blur + noise"),
            ("assumed", "Assumed blur: 1.0 px"), ("known", "Known blur: 1.6 px")
        ]):
            axes[row, col].imshow(example[key], vmin=0, vmax=1)
            axes[row, col].set_title(f"{example['source_id']} | {title}", fontsize=10)
            axes[row, col].axis("off")
    fig.suptitle("First reconstruction pilot | Four development images", fontsize=17)
    fig.text(0.5, 0.01, "512 × 512 native-resolution centre crops; Gaussian sensor-noise case. Known blur is a diagnostic reference.",
             ha="center", fontsize=10)
    fig.tight_layout(rect=(0, .025, 1, .965))
    fig.savefig(output_dir / "reconstruction_overview.png", dpi=135)
    figures.append(fig)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey="row")
    curves = result["curves"]
    for row, scenario in enumerate(result["config"]["scenarios"]):
        for col, model in enumerate(["assumed_blur", "known_blur_diagnostic"]):
            axis = axes[row, col]
            selection = curves[(curves.scenario == scenario) & (curves.model == model)]
            for score in COLOURS:
                series = selection[selection.score == score].groupby("coverage")["mse"].mean()
                axis.plot(series.index * 100, series.values * 1000,
                          label=LABELS[score], color=COLOURS[score],
                          linestyle=STYLES[score], linewidth=2)
            axis.set_title(("Blur + noise" if row == 0 else "Blur + noise + JPEG") +
                           (" | Assumed blur" if col == 0 else " | Known-blur diagnostic"), fontsize=11)
            axis.grid(axis="y", alpha=.2)
            axis.set_ylim(bottom=0)
            axis.set_xlim(10, 100)
            if row == 1:
                axis.set_xlabel("Pixels retained in each image (%)")
            if col == 0:
                axis.set_ylabel("Retained RGB MSE (× 10⁻³)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .94), ncol=3,
               fontsize=9, frameon=False)
    fig.suptitle("Selective reconstruction | Lower error is better", fontsize=16, y=.985)
    fig.text(.5, .015, "Equal-weight average of four source-image curves. Development only; no confidence intervals or calibration guarantee.",
             ha="center", fontsize=9)
    fig.tight_layout(rect=(0, .035, 1, .865))
    fig.savefig(output_dir / "risk_coverage.png", dpi=150)
    figures.append(fig)

    fig = plt.figure(figsize=(14, 7), layout="constrained")
    grid = fig.add_gridspec(2, len(examples) + 1,
                          width_ratios=[1] * len(examples) + [.055])
    for row, (key, title) in enumerate([("error", "Actual RGB RMSE"), ("spread", "Operator perturbation spread")]):
        vmax = max(np.max(example[key]) for example in examples)
        for col, example in enumerate(examples):
            axis = fig.add_subplot(grid[row, col])
            plotted = axis.imshow(example[key], cmap="magma", vmin=0, vmax=vmax)
            axis.set_title(f"{example['source_id']} | {title}", fontsize=9)
            axis.axis("off")
        fig.colorbar(plotted, cax=fig.add_subplot(grid[row, -1]),
                     label="RGB intensity units [0,1]")
    fig.suptitle("Error and operator sensitivity | Assumed-blur reconstruction", fontsize=16)
    fig.supxlabel("Each row has a shared colour scale. Spread measures reconstruction changes; it is not a probability or confidence interval.",
                  fontsize=9)
    fig.savefig(output_dir / "error_sensitivity_maps.png", dpi=150)
    figures.append(fig)
    return figures


In [17]:
figures = plot_results(result, OUTPUT_DIR)
plt.show()


Output hidden; open in https://colab.research.google.com to view.

### 9. Review checks and reproduction details
All original source hashes must remain unchanged. Every risk curve must reconcile with full-image risk at 100% coverage,
and oracle error sorting must be at least as good as each other selection rule at equal coverage.


In [18]:
print(json.dumps(result["checks"], indent=2))
print("\nExecution environment:")
print(json.dumps(result["environment"], indent=2))


{
  "constant_preserved": true,
  "adjoint_check": true,
  "ridge_normal_equation_max_abs": 2.8189256484623115e-16,
  "selection_endpoint_check": true,
  "psnr_scale_check": true,
  "source_files_unchanged": true,
  "no_duplicate_rgb_sources": true,
  "all_curves_reconcile_at_full_coverage": true,
  "oracle_lower_bound_check": true,
  "source_images": 4,
  "metrics_rows": 24,
  "scope": "four development images; no independent inference"
}

Execution environment:
{
  "python": "3.13.15",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scipy": "1.16.3",
  "pillow": "11.3.0",
  "matplotlib": "3.10.0"
}


## Takeaways
- The wrong blur assumption worsened this fixed inverse relative to the known-blur diagnostic.
- The degraded observation outperformed both fixed inverses in pooled MSE; noise amplification and regularisation need attention.
- Operator sensitivity provided some error ranking, but measurement residual won the blur/noise comparison at 50% retention;
  the image-gradient heuristic won the JPEG comparison at that coverage.
- Sensitivity can also rank errors when the blur is known. Positive ranking alone does not demonstrate mismatch specificity.
- No pretrained deep reconstruction, calibrated confidence, generative hallucination detection, or independent generalisation has been demonstrated.


### Next experiment
1. Keep 0801–0804 as development sources. Freeze this run as `00`; record all later changes as a new experiment.
2. On development data only, compare regularisation strengths and reconstruction baselines across more than one blur/noise setting.
   Keep a degraded-input baseline and operator perturbations fixed before final evaluation.
3. Add a pretrained physics-based reconstruction method and a genuine image-only uncertainty baseline,
   documenting their training data and checkpoint versions. Separate sensitivity to the operator from sensitivity to noise,
   image gradients and regularisation. Whole-image MSE alone does not establish detail reliability or hallucination detection.
4. Finalise at least 100 source-disjoint calibration images and at least 100 untouched test images from eligible sources.
   Account for development use and possible pretrained-model exposure. The current validation archive alone is too small for that target.
5. Fit any selection thresholds on calibration data only. Then evaluate risk–coverage, error localisation and source-level uncertainty
   on the locked test set. Crops from one source remain in one partition; bootstrap sources rather than pixels.

### References and scope
- [Official DIV2K dataset, structure and research-use notice](https://data.vision.ee.ethz.ch/cvl/DIV2K/).
- Agustsson, E. & Timofte, R. (2017). *NTIRE 2017 Challenge on Single Image Super-Resolution: Dataset and Study.* CVPR Workshops.
- Timofte et al. (2017). *NTIRE 2017 Challenge on Single Image Super-Resolution: Methods and Results.* CVPR Workshops.
Use and cite the images according to the official dataset terms. The images remain copyright of their owners.

**Validation method:** the supplied code cells were executed sequentially in one Python process, with stdout and figures
captured into this notebook. Native Jupyter-kernel execution was unavailable in the authoring environment
(nbformat/nbclient/ipykernel could not be installed). The structure received local v4-field checks, not nbformat-schema validation.
To close that environment check, open this notebook in JupyterLab or Colab and choose **Run all**.


## 10. Save this run to Google Drive
Added after reviewing the saved Colab run on 17 September 2026. The earlier cells write to a temporary `results` folder. Run the cell below **in the same live session** to copy all CSVs, checks and figures into a new dated folder in your project Drive. Existing runs are preserved.

The earlier saved output reports successful numerical checks; this export cell has not yet been run in your Colab session. If the session has ended, run the notebook from the beginning first. The Colab figure output is externally embedded and was not available for visual inspection through the Drive connector.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import shutil

# Run this cell in the same Colab session after the pilot and figures finish.
def save_pilot_results_to_drive(source_dir, project_dir):
    source_dir, project_dir = Path(source_dir), Path(project_dir)
    names = [
        "manifest.csv", "metrics.csv", "risk_coverage.csv",
        "score_correlations.csv", "summary.csv", "config.json",
        "checks.json", "environment.json", "reconstruction_overview.png",
        "risk_coverage.png", "error_sensitivity_maps.png",
    ]
    missing = [name for name in names if not (source_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(
            "Results missing: " + ", ".join(missing)
            + ". Run the pilot and plotting cells first in this session."
        )
    if not project_dir.is_dir():
        raise FileNotFoundError("Mount Google Drive and check the project folder first.")
    checks = json.loads((source_dir / "checks.json").read_text())
    required_checks = [
        "constant_preserved", "adjoint_check", "selection_endpoint_check",
        "psnr_scale_check", "source_files_unchanged", "no_duplicate_rgb_sources",
        "all_curves_reconcile_at_full_coverage", "oracle_lower_bound_check",
    ]
    if not all(checks.get(key) is True for key in required_checks):
        raise ValueError("The saved numerical checks have not all passed.")
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    destination = project_dir / "results" / ("pilot_00_" + timestamp)
    destination.mkdir(parents=True, exist_ok=False)
    manifest = []
    for name in names:
        source, target = source_dir / name, destination / name
        before = hashlib.sha256(source.read_bytes()).hexdigest()
        shutil.copy2(source, target)
        after = hashlib.sha256(target.read_bytes()).hexdigest()
        if before != after:
            raise IOError("Copy verification failed: " + name)
        manifest.append({"name": name, "size_bytes": target.stat().st_size,
                         "sha256": after})
    (destination / "export_manifest.json").write_text(json.dumps({
        "experiment": "pilot_00", "role": "development_only",
        "saved_at_utc": timestamp, "copied_files": manifest,
        "all_copy_hashes_match": True,
    }, indent=2) + "\n")
    return destination

SAVED_RESULTS_DIR = save_pilot_results_to_drive(
    OUTPUT_DIR,
    Path("/content/drive/MyDrive/reliable-reconstruction-under-mismatch"),
)
print("Saved and verified 11 result files in:", SAVED_RESULTS_DIR)
